In [0]:
%sql
CREATE OR REPLACE TABLE silver.customers
USING DELTA
AS
SELECT
    customer_key,
    customer_id,
    customer_number,

    -- Remove extra spaces from names
    TRIM(first_name) AS first_name,
    TRIM(last_name) AS last_name,

    -- Create a full name column
    CONCAT(
        TRIM(first_name),
        ' ',
        TRIM(last_name)
    ) AS full_name,

    -- Standardise country values
    CASE
        WHEN country IS NULL THEN 'Unknown'
        WHEN TRIM(country) = '' THEN 'Unknown'
        WHEN LOWER(TRIM(country)) = 'n/a' THEN 'Unknown'
        ELSE INITCAP(TRIM(country))
    END AS country,

    -- Standardise marital status
    INITCAP(TRIM(marital_status)) AS marital_status,

    -- Standardise gender values
    CASE
        WHEN gender IS NULL THEN 'Not Specified'
        WHEN TRIM(gender) = '' THEN 'Not Specified'
        WHEN LOWER(TRIM(gender)) = 'n/a' THEN 'Not Specified'
        ELSE INITCAP(TRIM(gender))
    END AS gender,

    -- Convert birthdate to DATE
    CAST(birthdate AS DATE) AS birthdate,

    -- Calculate customer age
    FLOOR(
        DATEDIFF(
            CURRENT_DATE(),
            CAST(birthdate AS DATE)
        ) / 365.25
    ) AS age_years,

    -- Convert create_date to DATE
    CAST(create_date AS DATE) AS create_date

FROM workspace.bronze.customers;

In [0]:
%sql
-- Verify: check no NULLs remain in country or gender
SELECT
  COUNT(*) AS total_customers,
  SUM(CASE WHEN country = 'Unknown' THEN 1 ELSE 0 END) AS unknown_country,
  SUM(CASE WHEN gender = 'Not Specified' THEN 1 ELSE 0 END) AS unspecified_gender
FROM workspace.silver.customers;